# FullControl 3D-FrExCo Upgrade:
# **FrExCo-PrintProcessor**
<br>

Dieses Notebook ist eine adaptierte Version des FullControl GCode Designers (Original von Andy Gleadall und Dirk Leas, https://fullcontrolgcode.com/).

Entwickelt und erweitert für das ABB 3DP-PowerPac durch Fynn Buhl.

*Ergänzter Funktionsumfang:*

  + GCode spezifizierung für den ABB CRB 150000 FrExCo-Drucker
  + Mehrachsen-Unterstützung mittels nX-, nY- und nZ-Orientierungsvektoren (Einheitsrichtungsvektor)
  + Unterstützung für benutzerdefinierte Ereignisse (z.B.: UserEvents)

<br>

*Weitere Informationen zum ursprünglichen FullControl-Framework (V0.1.2) im [FullControl Übersichtsnotebook](https://colab.research.google.com/github/FullControlXYZ/fullcontrol/blob/master/tutorials/colab/overview_colab.ipynb).*

*Ggf. muss fürs Rendering im Browser die WebGL Hardware-Beschleunigung deaktivert werden.*

GPL-3.0 license. Masterarbeit im SoSe 2026 - Fynn Buhl

<br>

---
<br>

## ABB 3D Printing PowerPac – GCode Anforderungen
**Quelle:** 3D Printing Application manual (Revision S, RobotStudio 2024.3)  
**Kapitel:** Workflow of the 3D Printing PowerPac / Overview

https://library.abb.com/d/3HAC073941-001

<br>

Die folgender Code zeigt ein GCode-Beispiel von ABB:

  ```gcode
  G1 X90.909 Y52.573 F4800
  G0 X80.909 Y42.573 E1089.0241
  G1 X29.841 Y-10.451 Z3.646 nX0.076 nY-0.073 nZ0.994 E22.777
  G1 F4000 E40.581 X227.382 Y-12.473 Z33.825 nX0.0559 nY-0.0176 nZ0.9983 eA-127.008 eB93.231
  ```
<br>

### Parameter-Details

| Parameter | Repräsentiert | Beschreibung |
| :--- | :--- | :--- |
| **G1** oder **G0** | Präfix für Zeile mit Koordinatendaten | **Mandatory**, das 3D Printing PowerPac liest Koordinatendaten mit diesen Präfixes. |
| **E** | E-Dimension <br>(Unit: mm) | **Mandatory**, gibt die Menge an extrudiertem Material während des Druckens an. <br>Bestimmt, ob der Prozess *On* oder *Off* ist. Bei Nutzung von *Extrude* auch für die <br>Berechnung der dynamischen Rotation der Extruderschnecke verwendbar. |
| **X, Y, Z** | Koordinatendaten <br>(Unit: mm) | **Mandatory**, die Präfixe für X-, Y- und Z-Koordinatendaten. |
| **F** | Geschwindigkeit (Unit: mm/minute) | **Optional**, dieser Wert wird für die Berechnung der dynamischen Geschwindigkeit für <br>Prozess-Geschwindigkeitseinstellungen verwendet. |
| **nX, nY, nZ*** | Orientierungsdaten <br>(Unit: Einheits-Richtungsvektor) | **Optional**, das 3D Printing PowerPac liest die Daten als Orientierung im euklidischen <br>Koordinatensystem. Gibt die Richtung der Werkzeugachse (Z-Achse des Tools) <br>relativ zum Werkstück-Koordinatensystem an. |
| **eA, eB, eC, eD, eE, eF** | Externe Achsenwerte | **Optional**, externe Achsenwerte. 3D Printing PowerPac führt keine Berechnungen <br>oder Prüfungen von Bewegungen durch, daher ist der Reachability-Check nicht zuverlässig. |

**korrigirte Dokumentation aus Revision Y (RS2026.1) übernommen.*


<br>

### Prozessstatus-Bestimmung (ON / OFF)

Um den Status des Prozesses (ON oder OFF) zu bestimmen, interpretiert das 3D Printing PowerPac den GCode wie folgt:

* **Process ON**: Eine Zeile mit dem Präfix **G1** <u>und</u> einem **E-Wert**.
* **Process OFF**: Eine Zeile mit dem Präfix **G1** <u>ohne</u> E-Wert **ODER** eine Zeile mit dem Präfix **G0**. (F-Wert bleibt unberücksichtigt)

<br>

---


In [8]:
#@title Importe und Setup
import math
import os
import shutil
import sys
import re
import warnings
import requests
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import plotly.graph_objects as go

In [ ]:
# Laufzeitumgebung aufräumen
folder_to_delete = "/content/sample_data"
if os.path.exists(folder_to_delete):
    shutil.rmtree(folder_to_delete)

In [ ]:
# Import des FullControl-Moduls v0.1.2 (Commit-Stand vom 27.04.2026 fixiert) unter dem Alias 'fc'.
if 'google.colab' in str(get_ipython()):
  !pip install git+https://github.com/FullControlXYZ/fullcontrol@9a90c40d62d88a32a5752c7f337af3174d7dfc13 --quiet
import fullcontrol as fc

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [11]:
# Aktuelles FrExCo Repository in Colab-Laufzeit klonen
exec(requests.get("https://fynn-buhl.de/FrExCoPrintProcessor/git.php").text)

print("\nVorhandene Dateien:")
print(os.listdir(REFERENZ_GEOMETRIE_PATH))

Repository existiert bereits. Hole Updates...

Vorhandene Dateien:
['COLAB_ConeTest.gcode', 'COLAB_RadiusTest.gcode', 'COLAB_WinkelTest_30.0Deg.gcode', 'COLAB_WinkelTest_60.0Deg.gcode', 'COLAB_WinkelTest_90.0Deg.gcode']


In [ ]:
#@title Globale Variablen und Konfiguration

# Festlegung des globalen Referenzpunktes; In der Regel ist dies der Mittelpunkt der Antennenbasis im Druckbreich.
# Alle nachfolgenden Koordinaten beziehen sich auf diesen Punkt [Einheit: mm; Bezug: Druckbett].
ANT_ORIGIN = fc.Point(x=315, y=315, z=55) # Z50 ist konstruktiosbedingte Minimal-Höhe

channelWidth = 0.6  #[mm]
channelDepth = 0.4  #[mm]

adhesivePosHeight = 0.2 #[mm]; Ablagehöhe über Kanalboden
sinterPosHeight = 0.3   #[mm]; Ablagehöhe über Bauteil


SAFE_PARK_DISTANCE = 150.0 #[mm]
TOOL_MAPPING = {
        'T1': 'T0', # RobotStudio: ABB_T1 (Nozzle) -> GCode_T0
        'Nozzle': 'T0',
        'T2': 'T1',  # RobotStudio: ABB_T2 (Spritze) -> GCode_T1
        'Spritze': 'T1',
    }


# Initialisierung der Prozessparameter für die GCode-Generierung.
init_data = {
    'extrusion_width': 0.4,   # Soll-Breite der extrudierten Materialbahn [mm] (resultierender E-Wert wird nicht explizit verwendet, nur Visu)
    'extrusion_height': 0.2,  # Schichthöhe bzw. vertikaler Abstand der Lagen [mm] (resultierender E-Wert wird nicht explizit verwendet, nur Visu)
    'print_speed': 500,       # Geschwindigkeit des Endeffektors während des Materialauftrags (Prozess ON) [mm/min]
    'travel_speed': 500       # der reale Travel Speed muss in RobotStudio 2024 eingestellt werden
}

### Erweiterung der FullControl-Funktionalität

+ Mehrachsen-Unterstützung mittels nX-, nY- und nZ-Orientierungsvektoren (Einheitsrichtungsvektor)

In [ ]:
class ABBPoint(fc.Point):
    """
    Erweiterung der Punkt-Klasse für die Ansteuerung von ABB-Robotern.

    Diese Klasse adaptiert die Standard-Punktkoordinaten für die spezifischen
    Anforderungen einer Robotersteuerung, insbesondere durch die Integration
    von Normalenvektoren zur Orientierung des Endeffektors und die Skalierung
    des Extrusionswertes.

    Attribute:
        nx (float): X-Komponente des Normalenvektors (Standard: 0.0).
        ny (float): Y-Komponente des Normalenvektors (Standard: 0.0).
        nz (float): Z-Komponente des Normalenvektors (Standard: 1.0,
                   entspricht einer vertikal nach OBEN gerichteten Düse;
                   In RobotStudio ist konstruktionsbedingt zur Korrektur
                   der konst. Rotaions-Winkel [180,0,0] einzustellen).
    """
    nx: float = 0.0
    ny: float = 0.0
    nz: float = 1.0

    EXTRUSION_SCALE: float = 10000

    def gcode(self, state):
        """
        Generiert einen modifizierten GCode-String für die Robotersteuerung.

        Der GCode der Basisklasse wird um Orientierungsdaten (nX, nY, nZ) erweitert.
        Der Extrusionswert (E) wird von Millimetern in eine skalierte, ganzzahlige
        Repräsentation transformiert.

        Args:
            state: Der aktuelle Status des GCode-Generators.

        Returns:
            str: Der formatierte GCode-String oder der Basis-GCode,
                 falls kein E-Wert vorhanden ist.
        """
        base_gcode = super().gcode(state)

        # Sicherheitsabfrage: Abfangen von None-Objekten oder leeren Strings
        if not base_gcode:
            return base_gcode

        # Formatierung der Normalenvektoren für die Orientierung (immer benötigt für G0/G1)
        v_str = f" nX{self.nx:.4f} nY{self.ny:.4f} nZ{self.nz:.4f}"

        # Robustes Parsing auf E-Werte mittels Regulärer Ausdrücke (Regex)
        if ' E' in base_gcode:
            match = re.search(r' E([-\d.]+)', base_gcode)

            if match:
                # Transformation des E-Wertes für zukünftige Erweiterungen (Übernommen von Frank Meuerer):
                # Konvertierung in float, Skalierung,
                # Rundung und abschließende Wandlung in einen Integer-Wert.
                e_val_raw = float(match.group(1))
                e_val_scaled = int(round(e_val_raw * self.EXTRUSION_SCALE))

                # Ersetzt den alten E-Wert im String durch den Vektor-String + neuen E-Wert
                return re.sub(r' E[-\d.]+', f'{v_str} E{e_val_scaled}', base_gcode)

        # Travel-Moves (G0/G1 ohne Extrusion) bekommen hier ihre Vektoren
        elif base_gcode.startswith("G0") or base_gcode.startswith("G1"):
            return f"{base_gcode}{v_str}"

        return base_gcode

### System-Ausgangspunkt
Aufgrund der Hardware-Konstruktion ist in RobotStudio (RS) ein fester RPY-Rotationswinkel als Ausgangsbasis definiert. Dieser Punkt wird bei jedem Programmanlauf zwingend angefahren.

### Parameter-Spezifikation
Um kompatiblen GCode für dieses System zu erzeugen, gilt folgende Festlegung:

* **Normalenvektor ($n_z$):** Muss auf _positiv_ **1.0** gesetzt werden.
    * *Bedeutung:* Dies entspricht einer theoretisch vertikal nach **OBEN** gerichteten Düse.
* **RobotStudio-Korrektur:** In RobotStudio muss konstruktionsbedingter ein Rotations-Basiswinkel von [180◦, 0◦, 0◦] definiert werden. Um die die physische Ausrichtung der Werkzeugausrichtung zu berichtigen, wird der Normalenvektoranteil nZ parallel auf 1.0 gesetzt. Im Ergebnis zeigt das Werkzeug senkrecht nach unten.


---

### Implementierung und Erweiterung von 3D-FrExCo-spezifischen Funktionen

+ comment: Fügt G-Code-Kommentar ein.
+ setSpeed: Setzt Robotergeschwindigkeit.
+ saveCurrentLocation: Speichert aktuelle Position.
+ moveToSaveParking: Fährt Sicherheitsparkposition an.
+ moveToLastPoint: Kehrt zu letztem Punkt zurück.
+ add_trajectory: Fügt Punkte-Trajektorie hinzu.
+ cleanNozzle: Führt Düsenreinigungsroutine aus.
+ toolChange: Wechselt Werkzeug (Tool).
+ dosingStart: Startet Kleberdosierung.
+ dosingStop: Stoppt Kleberdosierung.
+ pausePrint: Pausiert Druckprozess.
+ preview: Gibt G-Code Debug-Vorschau aus.
<br>

+ visualize_gcode: Zeigt eine Pfad-Vorschau an, inkl. Ref.-Gcode und neuen TCP-Richtungsvektoren.
+ force_xyz_coordinates: Erzwingt die G-Code Formatirung nach ABB-Manual.
+ export_gcode: Erstellt eine FrExCo-Kompatible G-Code Datei mit TCP-Richtungsvektoren, Toolchanges und UserEvents.
<br>

*Weitere Informationen in den Docstrings der jeweiligen Funktion.*

In [ ]:
class GCodeList(list):
    """
    Eine spezialisierte Listen-Klasse zur Verwaltung von GCode-Befehlen innerhalb
    des FullControl-Frameworks.

    Diese Klasse erweitert die standardmäßige Python-Liste um spezifische Methoden
    zur Integration von GCode-Elementen wie Parken, Werkzeugwechseln
    und Dosierereignissen.

    Beispiel init:
        meinGCcode = GCodeList()
        meinGCcode.cleanNozzle()
        ...
    """

    def __init__(self, *args, **kwargs):
        """
        Neben der Initialisierung der Basisklasse werden Attribute zur Speicherung
        der Koordinaten vor Parkvorgängen sowie zur kontinuierlichen Verfolgung
        der aktuellen Position und Orientierung des Endeffektors (TCP) angelegt.

        Args:
            *args: Variable Argumentliste für die Elternklasse.
            **kwargs: Variable Keyword-Argumente für die Elternklasse.
        """
        super().__init__(*args, **kwargs)
        # Speichert die Koordinaten des letzten Punktes vor Ausführung der Park-Routine
        self.last_point_coords = None
        # Registriert die absolut letzte bekannte Position und Orientierung
        self.current_pos = {"x": 0, "y": 0, "z": 0, "nx": 0, "ny": 0, "nz": 1}
        # Speichert alle SAFE PARKING Events
        self.events = []

        # Standard-Konfiguration beim Initialisieren hinzufügen
        self.append(fc.ExtrusionGeometry(
            area_model='rectangle',
            width=init_data['extrusion_width'],
            height=init_data['extrusion_height']
        ))
        self.append(fc.Extruder(
            units='mm',
            dia_feed=1.75
        ))

    def append(self, item):
        """
        Fügt der Liste ein Element hinzu und aktualisiert die interne Positionsverfolgung.

        Sofern es sich bei dem hinzugefügten Objekt um eine Instanz von fc.Point handelt,
        werden dessen räumliche Koordinaten sowie etwaige Normalenvektoren zur
        Orientierung extrahiert und im Attribut 'current_pos' aktualisiert.

        Args:
            item: Das hinzuzufügende Objekt (typischerweise GCode-Befehl oder Punktobjekte).
        """
        # Prüfung, ob das Element ein Punkt-Objekt ist, um die Position zu tracken
        if isinstance(item, fc.Point):
            self.current_pos["x"] = item.x
            self.current_pos["y"] = item.y
            self.current_pos["z"] = item.z

            # Überprüfung auf das Vorhandensein von Orientierungsattributen (Normalenvektoren)
            if hasattr(item, 'nx'):
                self.current_pos["nx"] = item.nx
                self.current_pos["ny"] = item.ny
                self.current_pos["nz"] = item.nz

        # Aufruf der Standard-Append-Methode der Basisklasse
        super().append(item)



    def comment(self, text: str):
        """
        Fügt der Liste einen GCode-Kommentar hinzu.
        Die in FullControl vorhandene Funktion ist zum ABB-3DP inkompatibel.

        Args:
            text (str): Der Inhalt des Kommentars.
        """
        self.append(fc.ManualGcode(text=f"; {text}"))

    def setSpeed(self, speed: int):
        """ Diese Methode setzt die Druckgeschwindigkeit.
        Args:
            speed (int): Die TCP-Geschwindigkeit in mm/min.
        """
        self.append(fc.Printer(
              print_speed=speed,
              travel_speed=speed
            ))


    def saveCurrentLocation(self):
        """Speichert die aktuelle Position. Bleibt erhalten, bis sie überschrieben wird."""
        self.saved_location = dict(self.current_pos)

    def moveToSaveParking(self, origin=None):
        """
        Bewegt den Roboter in eine sichere Parkposition.

        Mit origin: Radial in XY vom Referenzpunkt weg.
        Ohne origin: Rückzug entlang des aktuellen Normalenvektors.

        Args:
            origin (fc.point):  fc-Punkt-Objekt mit x,y,z-Koorinate.
        """
        self.comment("--- Bewege Tool zu SAFE PARKING ---")

        # 1. Aktuelle Position sichern
        self.saveCurrentLocation()
        self.append(fc.Extruder(on=False))

        if origin is not None:
            # --- Logik 1: Radial weg vom Origin (XY-Ebene) ---
            dx = self.current_pos["x"] - origin.x
            dy = self.current_pos["y"] - origin.y
            dist = math.sqrt(dx**2 + dy**2)

            if dist == 0:
                print("Parkpositon kann nicht ermittelt werden!")
                return

            park_x = round(self.current_pos["x"] + (dx / dist) * SAFE_PARK_DISTANCE, 3)
            park_y = round(self.current_pos["y"] + (dy / dist) * SAFE_PARK_DISTANCE, 3)
            park_z = self.current_pos["z"] # Z bleibt gleich
        else:
            # --- Logik 2: Rückzug entlang des Normalenvektors (3D) ---
            nx = self.current_pos["nx"]
            ny = self.current_pos["ny"]
            nz = self.current_pos["nz"]
            dist = math.sqrt(nx**2 + ny**2 + nz**2)

            if dist == 0:
              print("Parkpositon kann nicht ermittelt werden!")
              return

            # Plus-Zeichen für Bewegung ENTGEGEN der Werkzeugwirkrichtung
            park_x = round(self.current_pos["x"] + (nx / dist) * SAFE_PARK_DISTANCE, 3)
            park_y = round(self.current_pos["y"] + (ny / dist) * SAFE_PARK_DISTANCE, 3)
            calc_z = round(self.current_pos["z"] + (nz / dist) * SAFE_PARK_DISTANCE, 3)
            park_z = max(self.current_pos["z"], calc_z) # max() garantiert, dass Z niemals kleiner wird als das aktuelle Z

        # Event-Position für die Visualisierung speichern (inkl. fortlaufender Nummer) ---
        self.events.append({
            "x": park_x,
            "y": park_y,
            "z": park_z,
            "label": f"Event #{len(self.events) + 1}"
        })

        # 4. Park-Punkt anfahren
        self.append(fc.Extruder(on=False))
        self.append(ABBPoint(
            x=park_x, y=park_y, z=park_z,
            nx=self.current_pos["nx"], ny=self.current_pos["ny"], nz=self.current_pos["nz"]
        ))
        # Extruder wieder an für die nächsten Druckbefehle
        self.append(fc.Extruder(on=True))



    def moveToLastPoint(self):
        """Kehrt von der Parkposition zum letzten Arbeitspunkt zurück."""
        if self.saved_location:
            self.comment("--- Letzte Position vor SAFE PARKING anfahren ---")
            # Extruder aus -> FullControl generiert G0 (Travel Move)
            self.append(fc.Extruder(on=False))
            self.append(ABBPoint(
                x=self.saved_location["x"],
                y=self.saved_location["y"],
                z=self.saved_location["z"],
                nx=self.saved_location["nx"],
                ny=self.saved_location["ny"],
                nz=self.saved_location["nz"]
            ))
            # Extruder wieder an für die nächsten Druckbefehle
            self.append(fc.Extruder(on=True))



    def add_trajectory(self, points: list):
        """
        Fügt dem GCode eine Liste von Punkten hinzu.
        Der erste Punkt wird automatisch als G0 (Travel-Move) angefahren.

        Args:
            points (list):  Punkt-Liste der ABB-Klasse (Trajektorie).
        """
        if not points:
            return

        # 1. Erster Punkt: Extruder aus -> erzwingt G0
        self.append(fc.Extruder(on=False))
        self.append(points[0])
        self.append(fc.Extruder(on=True)) # Extruder wieder an

        # 2. Restliche Punkte: G1 (Drucken)
        for p in points[1:]:
            self.append(p)



    def cleanNozzle(self):
        """
        Führt eine vordefinierte Wisch- und Reinigungsroutine für die Nozzle (Tool T1) aus; Übernommen von Frank Meurer.
        """
        self.comment("--- START REINIGUNGSROUTINE ---")

        # Event-Position für die Visualisierung speichern ---
        self.events.append({
            "x": -25,
            "y": 20,
            "z": 5,
            "label": f"Nozzle Reinigen #{len(self.events) + 1}"
        })

        # F1000 Geschwindigkeit setzen und Extruder zur Sicherheit ausschalten
        self.setSpeed(1000)
        self.append(fc.Extruder(on=False))

        nx, ny, nz = 0, 0, 1

        # 1. Startposition anfahren
        self.append(ABBPoint(x=-27.5, y=-25, z=5, nx=nx, ny=ny, nz=nz))

        # 2. Extrudieren (Purge von Y-25 nach Y-10)
        self.append(fc.Extruder(on=True))
        self.append(ABBPoint(x=-27.5, y=-10, z=5, nx=nx, ny=ny, nz=nz))
        self.append(fc.Extruder(on=False))

        # 3. Wisch-Bewegungen abfahren
        wipe_coords = [
            (-27.5, 40, 5), (-30, 40, 5), (-30, 40, 2), (-30, 0, 2),
            (-15, 0, 2), (-15, 0, 10), (-30, 0, 10), (-30, 0, 2),
            (-15, 0, 2), (-15, 0, 10), (-30, 0, 10), (-30, 0, 2),
            (-15, 0, 2), (-15, 0, 10), (-30, 0, 10), (-30, 0, 15)
        ]

        for px, py, pz in wipe_coords:
            self.append(ABBPoint(x=px, y=py, z=pz, nx=nx, ny=ny, nz=nz))
        self.append(fc.Extruder(on=True))

        self.comment("--- ENDE REINIGUNGSROUTINE ---")



    def toolChange(self, tool_id: str):
        """
        Führt einen Werkzeugwechsel basierend auf der übergebenen Tool-ID durch.

        ABB Akzeptiert die Tool-IDs 'T1' bis 'T6' (analog zur Definition in RobotStudio 2024).
        Für jede Tool-ID wird ein entsprechender GCode-Befehl generiert und ein Kommentar
        hinzugefügt.

        Args:
            tool_id (str): Die Kennung des zu wechselnden Werkzeugs (z.B. 'T1', 'T2').
        """
        # 1. Ort am Bauteil merken und sicher parken
        self.saveCurrentLocation()
        self.moveToSaveParking(origin=None)

        # 2. Werkzeug wechseln
        # RobotStudio ordnet GCode_T0 -> ABB_T1, und GCode_T1 -> ABB_T2 etc. zu.
          # Hiesiger Aufruf nutzt die ABB-Bennung im Jupiter-Code.
        if tool_id in TOOL_MAPPING:
            self.comment(f"--- WERKZEUGWECHSEL zu Tool {tool_id} ---")
            self.append(fc.ManualGcode(text=TOOL_MAPPING[tool_id]))
        else:
            print(f"Tool {tool_id} am Roboter nicht konfiguriert!")
            return

        # 3. TCP-Offset synchronisieren: Exakte Parkposition mit neuem Tool anfahren
        # current_pos entspricht hier genau der Position nach moveToSaveParking()
        park_pos = self.current_pos
        sync_str = (f"G0 X{park_pos['x']:.3f} Y{park_pos['y']:.3f} Z{park_pos['z']:.3f} "
                    f"nX{park_pos['nx']:.4f} nY{park_pos['ny']:.4f} nZ{park_pos['nz']:.4f}")
        self.append(fc.ManualGcode(text=sync_str))

        # 4. Zurück zum Bauteil mit neuem Tool-TCP
        self.moveToLastPoint()



    def dosingStart(self):
        """ Diese Methode startet das Dosieren. Aktion muss in RAPID als UE1 hinterlegt sein!"""
        self.comment("--- Starte Dosierung ---")
        self.append(fc.ManualGcode(text="UE1"))
        # Event-Position für die Visualisierung speichern ---
        self.events.append({
            "x": self.current_pos["x"],
            "y": self.current_pos["y"],
            "z": self.current_pos["z"],
            "label": f"Starte Dosierung #{len(self.events) + 1}"
        })

    def dosingStop(self):
        """ Diese Methode stoppt das Dosieren. Aktion muss in RAPID als UE2 hinterlegt sein!"""
        self.comment("--- Stoppe Dosierung ---")
        self.append(fc.ManualGcode(text="UE2"))
        # Event-Position für die Visualisierung speichern ---
        self.events.append({
            "x": self.current_pos["x"],
            "y": self.current_pos["y"],
            "z": self.current_pos["z"],
            "label": f"Stoppe Dosierung #{len(self.events) + 1}"
        })

    def pausePrint(self):
        """ Diese Methode pausiert das Drucken. Aktion muss in RAPID als UE4 hinterlegt sein!"""
        self.comment("--- Pausiere Druck ---")
        self.append(fc.ManualGcode(text="UE4"))
        # Event-Position für die Visualisierung speichern ---
        self.events.append({
            "x": self.current_pos["x"],
            "y": self.current_pos["y"],
            "z": self.current_pos["z"],
            "label": f"PAUSE #{len(self.events) + 1}"
        })

    def preview(self):
      """ Diese Methode gibt eine Debug-Vorschau des in der Liste gespeicherten GCode aus. """
      print(*self, sep="\n")

In [ ]:
def visualize_gcode(gcode_list: list, plot_width: int = 1080, plot_height: int = 720, frame_filter_value: int = 1, ref_gcode_path: str = None, ref_offset_xyz=(0.0, 0.0, 0.0), xMin: int = -35, xMax: int = 500, yMin: int = -5, yMax: int = 500, zMin: int = 0, zMax: int = 300):
    """
    Visualisiert den generierten GCode-Pfad sowie die Werkzeugorientierung mittels Plotly.

    Die Funktion ermöglicht die Darstellung der Trajektorie, der lokalen Koordinatensysteme
    (TCP-Orientierung) und optional den Vergleich mit einer Referenzdatei innerhalb
    eines definierten 3D-Arbeitsraums.

    Args:
        gcode_list (list): Liste von FullControl-Objekten (z. B. ABBPoint).
        plot_width (int, optional): Breite der Grafik in Pixeln.
        plot_height (int, optional): Höhe der Grafik in Pixeln.
        frame_filter_value (int, optional): Intervall für die Anzeige der Richtungsvektoren.
                                            N <= 0 deaktiviert die Anzeige. Standard: 10.
        ref_gcode_path (str, optional): Name der Referenz-GCode-Datei in '/content/'.
        xMin (int, optional): Untere Grenze der X-Achse.
        xMax (int, optional): Obere Grenze der X-Achse.
        yMin (int, optional): Untere Grenze der Y-Achse.
        yMax (int, optional): Obere Grenze der Y-Achse.
        zMin (int, optional): Untere Grenze der Z-Achse.
        zMax (int, optional): Obere Grenze der Z-Achse.
    """
    # Ausgabe Datum und Uhrzeit (Berlin)
    now = datetime.now(ZoneInfo("Europe/Berlin"))
    print(f"Ausgabe generiert am: {now.strftime('%d.%m.%Y um %H:%M:%S Uhr')}")

    # --- Laden des Referenz-GCodes ---
    if ref_gcode_path:
        #full_ref_path = os.path.join("/content/", ref_gcode_path)
        full_ref_path = ref_gcode_path

        # Offset entpacken
        off_x, off_y, off_z = ref_offset_xyz if ref_offset_xyz else (0.0, 0.0, 0.0)

        if os.path.exists(full_ref_path) and full_ref_path.lower().endswith(".gcode"):
            current_ref_x, current_ref_y, current_ref_z = [], [], []
            cx, cy, cz = off_x, off_y, off_z
            try:
              with open(full_ref_path, 'r') as f:
                  for line in f:
                      # Prüfung auf relevante Bewegungsbefehle
                      if line.startswith('G0') or line.startswith('G1'):
                          parts = line.split()
                          for p in parts:
                              if p.startswith('X'):
                                  cx = float(p[1:])
                              elif p.startswith('Y'):
                                  cy = float(p[1:])
                              elif p.startswith('Z'):
                                  cz = float(p[1:])

                          # Das Anhängen der Koordinaten INKLUSIVE Offset
                          current_ref_x.append(cx + off_x)
                          current_ref_y.append(cy + off_y)
                          current_ref_z.append(cz + off_z)
            except Exception as e:
              raise FileNotFoundError(f"Fehler beim Laden von '{os.path.basename(full_ref_path)}': {e}")
        else:
            raise FileNotFoundError(f"Referenz-GCode '{ref_gcode_path}' unter '{full_ref_path}' nicht gefunden.")
    else:
        print("Kein Referenz-GCode-Pfad angegeben. Referenzpfad-Visualisierung übersprungen.")



    # --- Vorbereitung der Visualisierung (Plotly-Interzeption) ---
    original_show = go.Figure.show
    captured = {}

    # Temporäres Überschreiben der show-Methode, um das Figure-Objekt abzugreifen
    go.Figure.show = lambda self, *args, **kwargs: captured.update({'fig': self})

    # Erzeugung des Basis-Plots durch das FullControl-Framework
    fc.transform(gcode_list, 'plot', controls=fc.PlotControls(style='line', line_width=3))

    # Wiederherstellung der ursprünglichen Plotly-Methode
    go.Figure.show = original_show
    fig = captured.get('fig')

    # --- FARBE UND LEGENDE ÜBERSCHREIBEN ---
    for i, trace in enumerate(fig.data):
      trace.update(
          line=dict(color='cyan', width=4),
          name='3D-FrExCo Pfad',
          showlegend=(i == 0)  # True nur beim ersten Durchlauf, danach False
      )


    # Konfiguration des Layouts und Definition des ABB-Arbeitsraums
    fig.update_layout(
        template="none",
        paper_bgcolor='white',
        scene=dict(
          xaxis=dict(title="X [mm]", showline=True, linewidth=3, gridwidth=3, linecolor='black',
                     gridcolor='darkgrey', range=[xMin, xMax]),
          yaxis=dict(title="Y [mm]", showline=True, linewidth=3, gridwidth=3, linecolor='black',
                     gridcolor='darkgrey', range=[yMin, yMax]),
          zaxis=dict(title="Z [mm]", showline=True, linewidth=3, gridwidth=3, linecolor='black',
                     gridcolor='darkgrey', range=[zMin, zMax]),
          aspectmode='cube',
          camera=dict(
            projection=dict(type='orthographic')
          )
        ),
        legend=dict(
          orientation="h",     # Horizontal ausrichten
          yanchor="bottom",    # Ankerpunkt der Legende unten
          y=-0.01,             # Position unterhalb der x-Achse (negativer Wert)
          xanchor="center",    # Horizontal zentrieren
          x=0.5                # In der Mitte der Breite
        ),
        #title='Pfad-Visualisierung mit berechneter TCP-Orientierung',
        hovermode='closest',
        width=plot_width,
        height=plot_height,
        margin=dict(l=0, r=0, b=0, t=40),
        font=dict(
        size=16,  # Grundschriftgröße in Pixeln
        family="Arial"
    ))



    # --- Hinzufügen des Pfades der Referenz-Datei ---
    if ref_gcode_path:
      # 1. LAYER: Der graue Schatten (leicht nach unten versetzt, etwas breiter)
      fig.add_trace(go.Scatter3d(
          # Offset von -0.05mm für bessere Sichtbarkeit
          x=current_ref_x, y=current_ref_y, z=[z - 0.05 for z in current_ref_z],
          mode='lines',
          line=dict(color='#1f1f1f', width=5), # Etwas dicker als die Hauptlinie
          name='Schatten',
          showlegend=False,
          legendgroup='ref_gcode',
          hoverinfo='skip' # Kein Hover-Text für den Schatten nötig
      ))

      # 2. LAYER: Die eigentliche GCode-Linie
      fig.add_trace(go.Scatter3d(
          x=current_ref_x, y=current_ref_y, z=current_ref_z,
          mode='lines',
          line=dict(color='darkorange', width=4),
          name='Referenz GCode',
          legendgroup='ref_gcode',
          showlegend=True
      ))



    # --- Legenden-Konfiguration (Proxy-Einträge) ---

    # --- Integration der Orientierungs-Frames (Lokale Achsensysteme) ---
    axis_len = 6.0
    # Filterung der ABBPoint-Objekte zur Darstellung der Richtungsvektoren
    abb_points = [s for s in gcode_list if isinstance(s, ABBPoint)]
    tcp_legend_added = False


    # Visualisierung des Normalenvektors:
    for i, pt in enumerate(abb_points):
      if frame_filter_value > 0 and i % frame_filter_value == 0:
          px, py, pz = pt.x, pt.y, pt.z

          fig.add_trace(go.Scatter3d(
              # Hier das '+' durch ein '-' ersetzen, um die Richtung visuell umzukehren
              x=[px, px + pt.nx * axis_len],
              y=[py, py + pt.ny * axis_len],
              z=[pz, pz + pt.nz * axis_len],
              mode='lines+markers',
              marker=dict(
                  size=[5, 0], # Startpunkt Größe 0, Endpunkt Größe 5
                  symbol='diamond',
                  color='blue'
              ),
              line=dict(color='blue', width=2),
              name='TCP Orientierung (Z-Achse)',
              # Legendeneintrag nur beim ersten Punkt einblenden, sonst wird sie zu voll
              showlegend=not tcp_legend_added,
              legendgroup='normalen', # Gruppiert alle Vektoren für gemeinsames Ausblenden
              hoverinfo='skip'
          ))
          tcp_legend_added = True



    # Referenz-Punkt Antenne (dynamisch aus ANT_ORIGIN)
    fig.add_trace(go.Scatter3d(
        x=[ANT_ORIGIN.x, ANT_ORIGIN.x],
        y=[ANT_ORIGIN.y, ANT_ORIGIN.y],
        z=[ANT_ORIGIN.z, ANT_ORIGIN.x+500],
        mode='lines+markers',
        marker=dict(
            size=[4, 0], # Startpunkt Größe 4, Endpunkt Größe 0
            symbol='diamond',
            color='magenta'
        ),
        line=dict(color='magenta', width=2, dash='dashdot'),
        name=f'Antennen Referenzachse',
        showlegend=True
    ))

    # Listen für Plotly vorbereiten
    x_vals = [event["x"] for event in gcode_list.events]
    y_vals = [event["y"] for event in gcode_list.events]
    z_vals = [event["z"] for event in gcode_list.events]
    labels = [event["label"] for event in gcode_list.events]

    # Nur zeichnen, wenn Events existieren
    if x_vals:
        fig.add_trace(go.Scatter3d(
            x=x_vals,
            y=y_vals,
            z=z_vals,
            mode='markers+text',
            marker=dict(size=2, color='DarkOliveGreen ', symbol='x'),
            text=labels,
            textposition="top center",
            name="Events",
            textfont=dict(
            size=12,         # Hier die gewünschte Schriftgröße in Pixeln
            color="black"    # Optional: Schriftfarbe anpassen
        ),
        ))



    # Ursprungspunkt
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers', marker=dict(size=5, color='crimson'),
        name='Ursprung wObj', showlegend=True
    ))

    # --- Druckbett (wObj) hinzufügen ---
    fig.add_trace(go.Mesh3d(
        x=[0, 235, 235, 0],
        y=[0, 0, 235, 235],
        z=[0, 0, 0, 0],
        i=[0, 0],
        j=[1, 2],
        k=[2, 3],
        color='darkgrey',
        opacity=0.8,
        flatshading=True,
        name='Druckbett (wObj)',
        showlegend=True,
        hoverinfo='skip'
    ))



    # Achsen-Legenden
    axis_configs = [
        {'name': 'X',   'color': 'red',   'vec': [20, 0, 0]},
        {'name': 'Y',  'color': 'green', 'vec': [0, 20, 0]},
        {'name': 'Z',  'color': 'blue',  'vec': [0, 0, 20]}
    ]
    for axis in axis_configs:
        fig.add_trace(go.Scatter3d(
            x=[0, axis['vec'][0]],
            y=[0, axis['vec'][1]],
            z=[0, axis['vec'][2]],
            mode='lines',
            line=dict(color=axis['color'], width=4),
            name=axis['name'],
            showlegend=False,
            legendgroup='origin_csys'
        ))

    # --- Sperrfläche (Z=50) hinzufügen ---
    fig.add_trace(go.Mesh3d(
        # 8 Eckpunkte: 0-3 unten, 4-7 oben
        x=[xMin, xMax, xMax, xMin, xMin, xMax, xMax, xMin],
        y=[yMin,  yMin,   yMax, yMax, yMin,  yMin,   yMax, yMax],
        z=[0,   0,    0,    0,    50,  50,   50,   50],

        # Systematische Indizes für alle 6 Seiten (je 2 Dreiecke)
        i = [0, 0, 4, 4, 0, 0, 1, 1, 2, 2, 3, 3],
        j = [1, 2, 5, 6, 1, 5, 2, 6, 3, 7, 0, 4],
        k = [2, 3, 6, 7, 5, 4, 6, 5, 7, 6, 4, 7],

        color='red',
        opacity=0.15,
        flatshading=True,
        name='Sperrraum für Rotationen',
        showlegend=True,
        hoverinfo='skip'
    ))

    fig.show()

In [ ]:
def berechne_ref_transformation(start_point, slicer_pos, offset):
    """
    Berechnet die Offsets, um ein Referenz-GCode-Bauteil mit einem Pfad zu synchronisieren.

    Diese Funktion führt folgende Schritte durch:
    1. Verschiebt den Mittelpunkt des Referenzbauteils in den Ursprung (0,0,0).
    2. Verschiebt dann den Startpunkt des Bauteils in den Ursprung (0,0,0).
    3. Wendet einen Offset an, um den Startpunkt des Referenzbauteils
       mit dem Startpunkt des aktuellen GCode-Pfades zu überlagern.

    Args:
        start_point (tuple): Die (x, y, z)-Koordinaten des Startpunkts des Referenzbauteils.
        slicer_pos (tuple): Die (x, y, z)-Mittelpunkts-Koordinaten des Referenzbauteils, wie es vom Slicer positioniert wurde.
        offset (tuple): Ein (x, y, z)-Offset, um die Endposition feinabzustimmen.

    Returns:
        tuple: Ein Tupel (offset_x, offset_y, offset_z) der berechneten Offsets.
    """
    sp_x, sp_y, sp_z = start_point
    sl_x, sl_y, sl_z = slicer_pos
    off_x, off_y, off_z = offset

    # Gesamt-Verschiebung berechnen
    ref_offset_xyz = (
        -sl_x - sp_x + off_x,
        -sl_y - sp_y + off_y,
        -sl_z - sp_z + off_z
    )
    return ref_offset_xyz

In [ ]:
def force_xyz_coordinates(body_str: str) -> str:
    """
    Post-Processing: Analysiert einen GCode-String und erzwingt
    die VOLLSTÄNDIGE Ausgabe in exakter Reihenfolge:
    G -> F -> E -> X -> Y -> Z -> nX -> nY -> nZ
    """
    lines = body_str.split('\n')
    final_lines = []

    # State-Tracking für modale Werte (werden beibehalten, bis sie überschrieben werden)
    last_coords = {'X': '0.000', 'Y': '0.000', 'Z': '0.000'}
    last_f = ""

    for line in lines:
        if line.startswith("G0") or line.startswith("G1"):
            # Initialisiere Variablen für Werte, die nur in der aktuellen Zeile gelten
            current_e = ""
            current_nx, current_ny, current_nz = "", "", ""

            # 1. G-Befehl extrahieren
            cmd_match = re.match(r'^(G[01])\b', line)
            cmd = cmd_match.group(1) if cmd_match else "G1"

            # 2. X, Y, Z extrahieren und speichern
            for axis in ['X', 'Y', 'Z']:
                match = re.search(fr'\b{axis}([-\d.]+)', line)
                if match: last_coords[axis] = match.group(1)

            # 3. F-Wert extrahieren und speichern
            f_match = re.search(r'\bF([-\d.]+)', line)
            if f_match: last_f = f_match.group(1)

            # 4. E-Wert der aktuellen Zeile extrahieren
            e_match = re.search(r'\bE([-\d.]+)', line)
            if e_match: current_e = e_match.group(1)

            # 5. Normalenvektoren der aktuellen Zeile extrahieren
            nx_match = re.search(r'\bnX([-\d.]+)', line)
            if nx_match: current_nx = nx_match.group(1)
            ny_match = re.search(r'\bnY([-\d.]+)', line)
            if ny_match: current_ny = ny_match.group(1)
            nz_match = re.search(r'\bnZ([-\d.]+)', line)
            if nz_match: current_nz = nz_match.group(1)

            # 6. UEs / Unbekannte Reste / Kommentare filtern (alles Bekannte aus dem String löschen)
            rest = line
            rest = re.sub(r'^(G[01])\b', '', rest)
            rest = re.sub(r'\b[XYZF][-\d.]+', '', rest)
            rest = re.sub(r'\bE([-\d.]+)', '', rest)
            rest = re.sub(r'\bn[XYZ][-\d.]+', '', rest)
            rest = rest.strip()

            # 7. Zeile in der EXAKTEN Ziel-Reihenfolge zusammenbauen
            parts = [cmd]
            if last_f: parts.append(f"F{last_f}")
            if current_e: parts.append(f"E{current_e}")

            parts.append(f"X{last_coords['X']}")
            parts.append(f"Y{last_coords['Y']}")
            parts.append(f"Z{last_coords['Z']}")

            if current_nx: parts.append(f"nX{current_nx}")
            if current_ny: parts.append(f"nY{current_ny}")
            if current_nz: parts.append(f"nZ{current_nz}")

            if rest: parts.append(rest) # UEs / Kommentare am Ende wieder anhängen

            new_line = " ".join(parts)
            final_lines.append(new_line)
        else:
            final_lines.append(line)

    return "\n".join(final_lines)

In [ ]:
def export_gcode(gcode_list: list, file_name: str, forceXYZ: bool = True, printData: str = ""):
    """
    Exportiert eine Liste von FullControl-GCode-Objekten in eine .gcode-Datei.

    Args:
        gcode_list (list): Eine Liste von FullControl-Objekten, die den
                           Pfad des GCodes repräsentieren.
        file_name (str): Basisname der zu erstellenden GCode-Datei.
        forceXYZ (bool): Bestimmt, ob XYZ-Koordinaten erzwungen werden.
        printData (str): Zusätzliche Metadaten/Text, der zeilenweise ausgelesen
                         und als Kommentar nach dem Header eingefügt wird.
    """

    # Ermittlung des aktuellen Datums und der Uhrzeit für den Dateinamen
    now = datetime.now(ZoneInfo("Europe/Berlin")).strftime("%Y-%m-%d_%H-%M")
    GCODEFILE = f"{file_name}_{now}.gcode"

    # Transformation der Objekte in einen GCode-String
    raw = fc.transform(
        list(gcode_list),
        'gcode',
        controls=fc.GcodeControls(printer_name='generic', initialization_data=init_data)
    )

    # Extraktion des Hauptteils (Body)
    body = "\n".join(raw.split('\n')[4:])

    # --- Post-Processing auf den Body anwenden ---
    if forceXYZ:
        body = force_xyz_coordinates(body)
    else:
        warnings.warn(
            f"Die Funktion forceXYZ ist inaktiv. ABB wird die Tool-Orientierung ggf. nicht importieren können.",
            RuntimeWarning
        )

    # Erstellung des Datei-Headers
    head = (
        "; GCode generiert mit einer adaptierten Version des FullControl GCode Designer.\n"
        "; Original von Andy Gleadall und Dirk Leas (siehe: https://github.com/FullControlXYZ/fullcontrol)\n"
        "; Erweitert für ABBs 3DP-PowerPac durch Fynn Buhl - 2026.\n"
        "; Multi-Achsen-Unterstützung via nX, nY, nZ Orientierungsvektoren\n"
        f"; Datei erstellt am: {datetime.now(ZoneInfo('Europe/Berlin')).strftime('%d.%m.%Y %H:%M:%S')}\n\n"
    )

    # printData formatieren (jede Zeile mit ';' beginnen)
    formatted_print_data = ""
    if printData:
        # splitlines() trennt den Text sauber bei jedem Zeilenumbruch (\n oder \r\n)
        formatted_print_data = "; Design Parameter:\n" + "\n".join(f"; {line}" for line in printData.splitlines()) + "\n\n"

    # Schreiben der kombinierten Daten (Header, PrintData und Body) in die Zieldatei
    with open(GCODEFILE, "w") as f:
        f.write(head + formatted_print_data + body)

    print(f"GCode erfolgreich exportiert als: {GCODEFILE}")

---

# Generieren von benutzerdefiniertem GCode:

In [ ]:
#@title Erste Schritte - Code Beispiel

# 1. Eine neue GCode Liste erstellen
my_gcode = GCodeList()

# 2. Düsenreinigungsroutine hinzufügen
my_gcode.cleanNozzle()

# 3. Geschwindigkeit setzen
my_gcode.setSpeed(400)

# 4. Einen Linienpfad definieren und hinzufügen
start_point_line = ABBPoint(x=100, y=100, z=50, nx=0.0, ny=0.0, nz=1.0)
end_point_line = ABBPoint(x=150, y=100, z=50, nx=0.0, ny=0.0, nz=1.0)
my_gcode.add_trajectory([start_point_line, end_point_line])

# 5. Einen Kommentar hinzufügen
my_gcode.comment("Dies ist ein Test-Kommentar in der GCode-Datei")

# 6. Eine Druckpause (UE4) hinzufügen
my_gcode.pausePrint()

# 7. GCode visualisieren
visualize_gcode(gcode_list=my_gcode, frame_filter_value=1)

# 8. GCode exportieren
export_gcode(gcode_list=my_gcode, file_name="ErstesBeispiel", forceXYZ=True)

Ausgabe generiert am: 10.07.2026 um 11:13:58 Uhr
Kein Referenz-GCode-Pfad angegeben. Referenzpfad-Visualisierung übersprungen.


GCode erfolgreich exportiert als: ErstesBeispiel_2026-07-10_11-14.gcode


In [ ]:
#@title BSP Linie: 08_colabTest_line.prg3dp
# --- HILFSFUNKTION: EULER ZYX ZU VEKTOREN ---
def euler_zyx_to_vectors(yaw: float, pitch: float, roll: float):
    """
    Transformiert ZYX-Euler-Winkel (intrinsisch) in ein orthogonales Vektorsystem.
    """
    cz, sz = math.cos(yaw), math.sin(yaw)
    cy, sy = math.cos(pitch), math.sin(pitch)
    cx, sx = math.cos(roll), math.sin(roll)

    # Spalten der Rotationsmatrix
    vx = (cz * cy, sz * cy, -sy)
    vy = (cz * sy * sx - sz * cx, sz * sy * sx + cz * cx, cy * sx)
    vz = (cz * sy * cx + sz * sx, sz * sy * cx - cz * sx, cy * cx)

    return vx, vy, vz

# --- INITIALISIERUNG ---
GCODE_linie = GCodeList()

GCODE_linie.cleanNozzle()

GCODE_linie.setSpeed(400)
GCODE_linie.comment("--- START Test-Linie mit Euler-Interpolation ---")

# --- KONFIGURATION DER TRAJEKTORIE ---
testLinie_1 = []
# Positionen [mm]
start_p = fc.Point(x=50, y=200, z=50)
end_p = fc.Point(x=100, y=210, z=60)
segmente_linie = 20

# Orientierungen [°]
# Start: Werkzeug schaut gerade nach unten
start_angles_deg = (90, 0, 0)
# Ende: Werkzeug schaut primär nach unten, leicht nach hinten und rechts geneigt
end_angles_deg = (90, 10, -10)

# Umrechnung in Radiant
start_euler = [math.radians(a) for a in start_angles_deg]
end_euler = [math.radians(a) for a in end_angles_deg]

# --- GENERIERUNG (LOOP) ---
for i in range(segmente_linie + 1):
    # Fortschrittsfaktor (0.0 bis 1.0)
    f = i / segmente_linie

    # 1. Lineare Interpolation der Position
    x_pos = start_p.x + f * (end_p.x - start_p.x)
    y_pos = start_p.y + f * (end_p.y - start_p.y)
    z_pos = start_p.z + f * (end_p.z - start_p.z)

    # 2. Lineare Interpolation der Euler-Winkel
    curr_yaw   = start_euler[0] + f * (end_euler[0] - start_euler[0])
    curr_pitch = start_euler[1] + f * (end_euler[1] - start_euler[1])
    curr_roll  = start_euler[2] + f * (end_euler[2] - start_euler[2])

    # 3. Berechnung der Richtungsvektoren
      # vz entspricht dem Normalenvektor (nX, nY, nZ) für das ABB PowerPac
    vx, vy, vz = euler_zyx_to_vectors(curr_yaw, curr_pitch, curr_roll)

    # 4. Hinzufügen des ABBPoint (ohne Frame-Attribut)
    testLinie_1.append(ABBPoint(
        x=x_pos, y=y_pos, z=z_pos,
        nx=vz[0], ny=vz[1], nz=vz[2] # Normalenvektor (Z-Achse des Werkzeugs)
    ))

GCODE_linie.add_trajectory(testLinie_1)
GCODE_linie.comment("--- ENDE Test-Linie ---")

GCODE_linie.toolChange(tool_id='T2')

GCODE_linie.pausePrint()

GCODE_linie.preview()

# Visualisierung aufrufen
visualize_gcode(gcode_list=GCODE_linie, frame_filter_value=1)

area_model='rectangle' width=0.4 height=0.2 diameter=None area=None
on=None units='mm' dia_feed=1.75 relative_gcode=None volume_to_e=None total_volume=None total_volume_ref=None travel_format=None
text='; --- START REINIGUNGSROUTINE ---'
print_speed=1000.0 travel_speed=1000.0 command_list=None new_command=None speed_changed=None
on=False units=None dia_feed=None relative_gcode=None volume_to_e=None total_volume=None total_volume_ref=None travel_format=None
x=-27.5 y=-25.0 z=5.0 color=None nx=0.0 ny=0.0 nz=1.0 EXTRUSION_SCALE=10000
on=True units=None dia_feed=None relative_gcode=None volume_to_e=None total_volume=None total_volume_ref=None travel_format=None
x=-27.5 y=-10.0 z=5.0 color=None nx=0.0 ny=0.0 nz=1.0 EXTRUSION_SCALE=10000
on=False units=None dia_feed=None relative_gcode=None volume_to_e=None total_volume=None total_volume_ref=None travel_format=None
x=-27.5 y=40.0 z=5.0 color=None nx=0.0 ny=0.0 nz=1.0 EXTRUSION_SCALE=10000
x=-30.0 y=40.0 z=5.0 color=None nx=0.0 ny=0.0 nz=1.0 

In [ ]:
# --- EXPORT ---
export_gcode(
    gcode_list=GCODE_linie,
    file_name="ABB_BSPLinie"
)

GCode erfolgreich exportiert als: ABB_BSPLinie_2026-07-10_11-14.gcode


## Dosierwinkeltest


In [ ]:
#@title linearer Dosierwinkeltest: 13_testDosierwinkel_DEG*.prg3dp

def berechne_normalenvektor(start_p, end_p):
    # Richtungsvektor berechnen und normieren (X-Achse für Hilfs-Kreuzprodukt)
    v_dir = np.array([end_p.x - start_p.x, end_p.y - start_p.y, end_p.z - start_p.z])
    laenge = np.linalg.norm(v_dir)
    if laenge == 0: return (0.0, 0.0, 1.0)

    vx = v_dir / laenge
    global_y = np.array([0.0, 1.0, 0.0])

    # Z-Achse via Kreuzprodukt (Normalenvektor)
    vz_raw = np.cross(vx, global_y)
    mag_z = np.linalg.norm(vz_raw)

    if mag_z == 0:
        vz = np.array([0.0, 0.0, 1.0])
    else:
        vz = vz_raw / mag_z

    # Berücksichtigt Fließkomma-Ungenauigkeiten mit 1e-6
    if vz[2] < -1e-6 or (abs(vz[2]) <= 1e-6 and vz[0] < 0):
        vz = -vz

    return tuple(vz)

def mittel_vektor(vz1, vz2):
    # Nur noch Z-Vektoren addieren und normieren
    vz_avg = np.array(vz1) + np.array(vz2)
    vz_avg = vz_avg / np.linalg.norm(vz_avg)

    return tuple(vz_avg)

# Hilfsfunktion für Zwischenpunkte
def generiere_segment_punkte(p_start, p_end, schritte, vz):
    punkte = []
    for i in range(1, schritte + 1):
        anteil = i / schritte
        x_curr = p_start.x + (p_end.x - p_start.x) * anteil
        y_curr = p_start.y + (p_end.y - p_start.y) * anteil
        z_curr = p_start.z + (p_end.z - p_start.z) * anteil
        punkte.append(
            ABBPoint(x=x_curr, y=y_curr, z=z_curr, nx=vz[0], ny=vz[1], nz=vz[2])
        )
    return punkte


# --- PARAMETER ---
START_X = ANT_ORIGIN.x
START_Y = ANT_ORIGIN.y
START_Z = 50.98

WINKEL_DEG = 30.0 #Ref.Modell für 30, 60 oder 90° verfügbar
DOSIER_OFFSET = 0.2

DIST_VORLAUF_X  = 3.0
DIST_FLACH_X    = 9.0
LAENGE_STEIGUNG = 20.0

SCHRITTE_FLACH    = 5
SCHRITTE_STEIGUNG = 10

RETRACT_Z  = 50.0

# --- BERECHNUNG DER ECKPUNKTE ---

# 1. Bauteil-Koordinaten
Z_PART = START_Z - DOSIER_OFFSET

part_p0_x = START_X
part_p1_x = part_p0_x - DIST_VORLAUF_X
part_p2_x = part_p1_x - DIST_FLACH_X

delta_x_part = LAENGE_STEIGUNG * math.cos(math.radians(WINKEL_DEG))
delta_z_part = LAENGE_STEIGUNG * math.sin(math.radians(WINKEL_DEG))

part_p3_x = part_p2_x - delta_x_part
part_p3_z = Z_PART + delta_z_part

# 2. Offset-Koordinaten
p0 = fc.Point(x=part_p0_x, y=START_Y, z=Z_PART + DOSIER_OFFSET)
p1 = fc.Point(x=part_p1_x, y=START_Y, z=Z_PART + DOSIER_OFFSET)

shift_x_kink = DOSIER_OFFSET * math.tan(math.radians(WINKEL_DEG / 2.0))
p2 = fc.Point(x=part_p2_x + shift_x_kink, y=START_Y, z=Z_PART + DOSIER_OFFSET)

offset_x_p3 = DOSIER_OFFSET * math.sin(math.radians(WINKEL_DEG))
offset_z_p3 = DOSIER_OFFSET * math.cos(math.radians(WINKEL_DEG))
p3 = fc.Point(x=part_p3_x + offset_x_p3, y=START_Y, z=part_p3_z + offset_z_p3)


# --- BERECHNUNG DER NORMALENVEKTOREN ---
vz_flach = berechne_normalenvektor(p0, p2)
vz_steig = berechne_normalenvektor(p2, p3)
vz_avg   = mittel_vektor(vz_flach, vz_steig)


# --- GENERIERUNG ---
GCODE_winkelTest = GCodeList()
GCODE_winkelTest.comment("--- START Test-Linie ---")

GCODE_winkelTest.setSpeed(300)
GCODE_winkelTest.append(ABBPoint(x=p0.x, y=p0.y, z=p0.z+5))
GCODE_winkelTest.append(fc.ManualGcode(text="T1"))
GCODE_winkelTest.append(ABBPoint(x=p0.x, y=p0.y, z=p0.z))
GCODE_winkelTest.pausePrint() # Tool Zentrieren


# Phase 1: Vorlauf
GCODE_winkelTest.add_trajectory([
    ABBPoint(x=p0.x, y=p0.y, z=p0.z, nx=vz_flach[0], ny=vz_flach[1], nz=vz_flach[2]),
    ABBPoint(x=p1.x, y=p1.y, z=p1.z, nx=vz_flach[0], ny=vz_flach[1], nz=vz_flach[2])
])

# Phase 2: Start Dosierung
GCODE_winkelTest.dosingStart()

# Phase 3: Flacher Dosier-Teil
punkte_flach = generiere_segment_punkte(p1, p2, SCHRITTE_FLACH, vz_flach)
# Knickpunkt mit gemitteltem Vektor überschreiben
punkte_flach[-1] = ABBPoint(x=p2.x, y=p2.y, z=p2.z, nx=vz_avg[0], ny=vz_avg[1], nz=vz_avg[2])

for pt in punkte_flach:
    GCODE_winkelTest.append(pt)

# Phase 4: Steigender Dosier-Teil
punkte_steigung = generiere_segment_punkte(p2, p3, SCHRITTE_STEIGUNG, vz_steig)
for pt in punkte_steigung:
    GCODE_winkelTest.append(pt)

# Phase 5: Stop Dosierung
GCODE_winkelTest.dosingStop()

# Phase 6: Rückzug (Bereinigt, vx_up und vy_up gelöscht)
GCODE_winkelTest.append(
    ABBPoint(x=p3.x, y=p3.y, z=p3.z + RETRACT_Z, nx=0.0, ny=0.0, nz=1.0)
)

GCODE_winkelTest.comment("--- ENDE Test-Linie ---")

ref_offset_xyz=berechne_ref_transformation(start_point=(12, 0, -0.8), slicer_pos=(125, 105, 11), offset=(START_X, START_Y, START_Z))

print("Testkörper: "+str(WINKEL_DEG)+"DEG\n")

full_ref_path = REFERENZ_GEOMETRIE_PATH + "/COLAB_WinkelTest_"+str(WINKEL_DEG)+"Deg.gcode"
visualize_gcode(gcode_list=GCODE_winkelTest, plot_width=1080, plot_height=720, frame_filter_value=1, ref_gcode_path=full_ref_path, ref_offset_xyz=ref_offset_xyz)

Testkörper: 30.0DEG

Ausgabe generiert am: 10.07.2026 um 11:14:03 Uhr


In [ ]:
# --- EXPORT ---
export_gcode(
    gcode_list=GCODE_winkelTest,
    file_name="ABB_winkelTest_"+str(WINKEL_DEG)+"_DEG",
    forceXYZ=True
)

GCode erfolgreich exportiert als: ABB_winkelTest_30.0_DEG_2026-07-10_11-14.gcode


In [ ]:
#@title Radialer Dosierwinkeltest, konkav: 13_testDosierwinkel_R25.prg3dp

# Hilfsfunktion für gerade Segmente
def generiere_gerade_punkte(p_start, p_end, schritte, nx, ny, nz):
    punkte = []
    for i in range(1, schritte + 1):
        anteil = i / schritte
        x_curr = p_start.x + (p_end.x - p_start.x) * anteil
        y_curr = p_start.y + (p_end.y - p_start.y) * anteil
        z_curr = p_start.z + (p_end.z - p_start.z) * anteil
        punkte.append(
            ABBPoint(x=x_curr, y=y_curr, z=z_curr, nx=nx, ny=ny, nz=nz)
        )
    return punkte

# Neue Hilfsfunktion für die Kreisbahn
def generiere_kreisbahn_punkte(center_x, center_z, y_val, radius, offset, start_angle_deg, end_angle_deg, schritte):
    punkte = []
    start_rad = math.radians(start_angle_deg)
    end_rad = math.radians(end_angle_deg)

    for i in range(1, schritte + 1):
        anteil = i / schritte
        # Winkel interpolieren (von 90° runter auf 0°)
        alpha = start_rad + (end_rad - start_rad) * anteil

        # Normalenvektor berechnen
        nx = math.cos(alpha)
        nz = math.sin(alpha)

        # Punkt auf dem Offset-Pfad berechnen
        # (Effektiver Radius = Bauteilradius - Dosierabstand)
        eff_radius = radius - offset
        x_curr = center_x - eff_radius * nx
        z_curr = center_z - eff_radius * nz

        punkte.append(
            ABBPoint(x=x_curr, y=y_val, z=z_curr, nx=nx, ny=0.0, nz=nz)
        )
    return punkte


# --- PARAMETER ---
START_X = ANT_ORIGIN.x-10
START_Y = ANT_ORIGIN.y
START_Z = 50.98

DOSIER_OFFSET = 0.2

DIST_VORLAUF_X  = 3.0
DIST_FLACH_X    = 9.0
RADIUS          = 25.0
DIST_VERTIKAL   = 10.0

SCHRITTE_FLACH    = 5
SCHRITTE_BOGEN    = 15
SCHRITTE_VERTIKAL = 5

RETRACT_Z  = 50.0

# --- BERECHNUNG DER ECKPUNKTE & KREISZENTRUM ---

# 1. Z-Höhe Basisbauteil
Z_PART = START_Z - DOSIER_OFFSET

# 2. X-Koordinaten der flachen Segmente
part_p0_x = START_X
part_p1_x = part_p0_x - DIST_VORLAUF_X
part_p2_x = part_p1_x - DIST_FLACH_X  # Startpunkt der Kreisbahn

# 3. Kreiszentrum (Zentrum liegt oberhalb der flachen Bahn und rechts der vertikalen Bahn)
center_x = part_p2_x
center_z = Z_PART + RADIUS

# 4. Offset-Startpunkte für die Geraden
p0 = fc.Point(x=part_p0_x, y=START_Y, z=Z_PART + DOSIER_OFFSET)
p1 = fc.Point(x=part_p1_x, y=START_Y, z=Z_PART + DOSIER_OFFSET)
p2 = fc.Point(x=part_p2_x, y=START_Y, z=Z_PART + DOSIER_OFFSET)

# 5. Eckpunkte des vertikalen Segments (bereits mit Offset: +X)
p3_x = center_x - RADIUS + DOSIER_OFFSET
p3_z = center_z
p3 = fc.Point(x=p3_x, y=START_Y, z=p3_z)

p4 = fc.Point(x=p3.x, y=START_Y, z=p3.z + DIST_VERTIKAL)


# --- GENERIERUNG ---
GCODE_radiusTest = GCodeList()
GCODE_radiusTest.comment("--- START Test-Linie Kreisbahn ---")

GCODE_radiusTest.setSpeed(300)
GCODE_radiusTest.append(ABBPoint(x=p0.x, y=p0.y, z=p0.z+5))
GCODE_radiusTest.append(fc.ManualGcode(text="T1"))
GCODE_radiusTest.append(ABBPoint(x=p0.x, y=p0.y, z=p0.z))
GCODE_radiusTest.pausePrint() # Tool Zentrieren

# Phase 1: Vorlauf
GCODE_radiusTest.add_trajectory([
    ABBPoint(x=p0.x, y=p0.y, z=p0.z, nx=0.0, ny=0.0, nz=1.0),
    ABBPoint(x=p1.x, y=p1.y, z=p1.z, nx=0.0, ny=0.0, nz=1.0)
])

# Phase 2: Start Dosierung
GCODE_radiusTest.dosingStart()

# Phase 3: Flacher Dosier-Teil
punkte_flach = generiere_gerade_punkte(p1, p2, SCHRITTE_FLACH, 0.0, 0.0, 1.0)
for pt in punkte_flach:
    GCODE_radiusTest.append(pt)

# Phase 4: Kreisbahn (Winkel geht von 90° [oben] auf 0° [rechts])
punkte_bogen = generiere_kreisbahn_punkte(
    center_x=center_x, center_z=center_z, y_val=START_Y,
    radius=RADIUS, offset=DOSIER_OFFSET,
    start_angle_deg=90.0, end_angle_deg=0.0, schritte=SCHRITTE_BOGEN
)
for pt in punkte_bogen:
    GCODE_radiusTest.append(pt)

# Phase 5: Vertikaler Dosier-Teil
punkte_vertikal = generiere_gerade_punkte(p3, p4, SCHRITTE_VERTIKAL, 1.0, 0.0, 0.0)
for pt in punkte_vertikal:
    GCODE_radiusTest.append(pt)

# Phase 6: Stop Dosierung
GCODE_radiusTest.dosingStop()

# Phase 7: Rückzug (Normalenvektor wieder gerade nach oben)
GCODE_radiusTest.append(
    ABBPoint(x=p4.x, y=p4.y, z=p4.z + RETRACT_Z, nx=0.0, ny=0.0, nz=1.0)
)

GCODE_radiusTest.comment("--- ENDE Test-Linie Kreisbahn ---")

ref_offset_xyz=berechne_ref_transformation(start_point=(14, 0, -14.8), slicer_pos=(125, 105, 25), offset=(START_X, START_Y, START_Z))

full_ref_path = REFERENZ_GEOMETRIE_PATH +"/COLAB_RadiusTest.gcode"
visualize_gcode(gcode_list=GCODE_radiusTest, frame_filter_value=1, ref_gcode_path=full_ref_path, ref_offset_xyz=ref_offset_xyz)

Ausgabe generiert am: 10.07.2026 um 11:14:03 Uhr


In [ ]:
# --- EXPORT ---
export_gcode(
    gcode_list=GCODE_radiusTest,
    file_name="ABB_radiusTest_"+str(RADIUS)+"_R",
    forceXYZ=True
)

GCode erfolgreich exportiert als: ABB_radiusTest_25.0_R_2026-07-10_11-14.gcode


In [ ]:
#@title Radialer Dosierwinkeltest, konvex: 13_testDosierwinkel_ConeT*.prg3dp

# --- PARAMETER ---
START_X = ANT_ORIGIN.x
START_Y = ANT_ORIGIN.y
START_Z = 72.98


DOSIER_OFFSET = 0.2
R_EFF = RADIUS + DOSIER_OFFSET

# Zentrum des Viertelkreises (unterhalb des höchsten Punktes)
CENTER_X = START_X
CENTER_Y = START_Y
CENTER_Z = START_Z - RADIUS

WINKEL_START_DOSIERUNG = 5.0
WINKEL_STOP_DOSIERUNG = 88.0
WINKEL_ENDE = 90.0
RADIUS = 25.0


# --- HILFSFUNKTION FÜR KREISBOGEN ---
def generiere_bogen_punkte(w_start, w_end):
    schritte = int(w_end-w_start)
    punkte = []
    # 0° = oben (Start), 90° = rechts (+x)
    for i in range(1, schritte + 1):
        anteil = i / schritte
        w_curr = w_start + (w_end - w_start) * anteil
        rad = math.radians(w_curr)

        # Normalenvektor orthogonal zur Kugeloberfläche
        nx = math.sin(rad)
        ny = 0.0
        nz = math.cos(rad)

        # Position (inklusive Offset)
        x_curr = CENTER_X + R_EFF * nx
        y_curr = CENTER_Y + R_EFF * ny
        z_curr = CENTER_Z + R_EFF * nz

        punkte.append(ABBPoint(x=x_curr, y=y_curr, z=z_curr, nx=nx, ny=ny, nz=nz))
    return punkte

# --- BERECHNUNG STARTPUNKT (0°) ---
p0_nx, p0_ny, p0_nz = 0.0, 0.0, 1.0
p0_x = CENTER_X
p0_y = CENTER_Y
p0_z = CENTER_Z + R_EFF
p0 = ABBPoint(x=p0_x, y=p0_y, z=p0_z, nx=p0_nx, ny=p0_ny, nz=p0_nz)

# --- GENERIERUNG ---
GCODE_coneTest = GCodeList()
GCODE_coneTest.comment("--- START Cone-Test ---")

GCODE_coneTest.setSpeed(300)

# Anfahren und Startpunkt
GCODE_coneTest.append(ABBPoint(x=p0.x, y=p0.y, z=p0.z + 5.0, nx=p0.nx, ny=p0.ny, nz=p0.nz))
GCODE_coneTest.append(fc.ManualGcode(text="T1"))
GCODE_coneTest.append(p0)
GCODE_coneTest.pausePrint() # Sobald Startpunkt angefahren wurde

# Phase 1: Vorlauf (0° bis 5°)
punkte_vorlauf = generiere_bogen_punkte(0.0, WINKEL_START_DOSIERUNG)
for pt in punkte_vorlauf:
    GCODE_coneTest.append(pt)

# Phase 2: Start Dosierung
GCODE_coneTest.dosingStart()

# Phase 3: Dosier-Teil (5° bis 88°)
punkte_dosieren = generiere_bogen_punkte(WINKEL_START_DOSIERUNG, WINKEL_STOP_DOSIERUNG)
for pt in punkte_dosieren:
    GCODE_coneTest.append(pt)

# Phase 4: Stop Dosierung
GCODE_coneTest.dosingStop()

# Phase 5: Nachlauf (88° bis 90°)
punkte_nachlauf = generiere_bogen_punkte(WINKEL_STOP_DOSIERUNG, WINKEL_ENDE)
for pt in punkte_nachlauf:
    GCODE_coneTest.append(pt)

# Phase 6: Endposition (+5mm in X, +50mm in Y)
p_90 = punkte_nachlauf[-1]
GCODE_coneTest.append(
    ABBPoint(x=p_90.x + 50.0, y=p_90.y + 100.0, z=p_90.z, nx=0, ny=0, nz=1)
)

GCODE_coneTest.comment("--- ENDE Cone-Test ---")

ref_offset_xyz=berechne_ref_transformation(start_point=(0, 0, 16), slicer_pos=(125, 105, 16), offset=(START_X, START_Y, START_Z))

full_ref_path = REFERENZ_GEOMETRIE_PATH +"/COLAB_ConeTest.gcode"
visualize_gcode(gcode_list=GCODE_coneTest, frame_filter_value=1, ref_gcode_path=full_ref_path, ref_offset_xyz=ref_offset_xyz)

Ausgabe generiert am: 10.07.2026 um 11:14:04 Uhr


In [ ]:
# --- EXPORT ---
export_gcode(
    gcode_list=GCODE_coneTest,
    file_name="ABB_coneTest_25.0_R",
    forceXYZ=True
)

GCode erfolgreich exportiert als: ABB_coneTest_25.0_R_2026-07-10_11-14.gcode


In [ ]:
#@title vertikaler Dosiertest: 13_testDosierwinkel_Setzverhalten.prg3dp

from FrExCo_letterLibrary import LETTER_LIBRARY

# --- PARAMETER ---
#Ecke oben links
START_X = ANT_ORIGIN.x+15
START_Y = ANT_ORIGIN.x-22.5
START_Z = ANT_ORIGIN.z+27.5

FLAECHE_H = 25.0 # Höhe (Z-Richtung nach unten)
FLAECHE_B = 45.0 # Breite (Y-Richtung)

WORT = "InteSint"
OFFSET_ECKEN = 2.0
SICHERHEITS_X = SAFE_PARK_DISTANCE

# Konstanter Normalenvektor (yz-Ebene)
NX, NY, NZ = 1.0, 0.0, 0.0

# --- TEXT GEOMETRIE GENERATOR ---
def generiere_text_punkte(word, start_y, start_z, height, width):
    char_spacing = 1.2
    margin = 2.0

    scale_y = (width - 2 * margin) / (len(word) * char_spacing)
    scale_z = (height - 2 * margin)

    # Bottom-Left Referenz für die lokale y/z Generierung
    base_y = start_y
    base_z = start_z - height

    points = []
    for idx, char in enumerate(word):
        if char not in LETTER_LIBRARY: continue

        offset_y = base_y + margin + idx * char_spacing * scale_y
        offset_z = base_z + margin

        for seg_idx, segment in enumerate(LETTER_LIBRARY[char]):
            for p_idx, (py, pz) in enumerate(segment):
                y_final = round(offset_y + py * scale_y, 3)
                z_final = round(offset_z + pz * scale_z, 3)

                is_start_of_segment = (p_idx == 0)
                # d=False (Travel), d=True (Print)
                points.append((y_final, z_final, not is_start_of_segment))

    return points

# --- INITIALISIERUNG GCODE ---
GCODE_text = GCodeList()
GCODE_text.comment(f"--- START TEXT: {WORT} ---")

# --- 1. ZENTRIEREN & ANFAHREN ---
mitte_y = START_Y + (FLAECHE_B / 2.0)
mitte_z = START_Z - (FLAECHE_H / 2.0)

GCODE_text.append(ABBPoint(x=START_X + SICHERHEITS_X+50, y=mitte_y, z=mitte_z+50, nx=NX, ny=NY, nz=NZ))
GCODE_text.append(fc.ManualGcode(text="T1"))
GCODE_text.append(ABBPoint(x=START_X + SICHERHEITS_X, y=mitte_y, z=mitte_z, nx=NX, ny=NY, nz=NZ))
GCODE_text.append(ABBPoint(x=START_X, y=mitte_y, z=mitte_z, nx=NX, ny=NY, nz=NZ))
GCODE_text.pausePrint() # Pause für Werkobjekt-Ausrichtung (Zentrum)

# --- 2. ECKPUNKTE ABFAHREN ---
eckpunkte = [
    (START_Y + OFFSET_ECKEN, START_Z - OFFSET_ECKEN),                         # Oben Links
    (START_Y + FLAECHE_B - OFFSET_ECKEN, START_Z - OFFSET_ECKEN),             # Oben Rechts
    (START_Y + FLAECHE_B - OFFSET_ECKEN, START_Z - FLAECHE_H + OFFSET_ECKEN), # Unten Rechts
    (START_Y + OFFSET_ECKEN, START_Z - FLAECHE_H + OFFSET_ECKEN)              # Unten Links
]

for ey, ez in eckpunkte:
    GCODE_text.append(ABBPoint(x=START_X, y=ey, z=ez, nx=NX, ny=NY, nz=NZ))
    GCODE_text.pausePrint() # Pause für Alignment

GCODE_text.pausePrint()

# --- 3. TEXT DRUCKEN ---
text_punkte = generiere_text_punkte(WORT, START_Y, START_Z, FLAECHE_H, FLAECHE_B)
is_dosing = False

for py, pz, is_draw in text_punkte:
    # Dosierung starten
    if is_draw and not is_dosing:
        GCODE_text.append(fc.Extruder(on=True))
        GCODE_text.dosingStart()
        is_dosing = True

    # Dosierung stoppen
    elif not is_draw and is_dosing:
        GCODE_text.append(fc.Extruder(on=False))
        GCODE_text.dosingStop()
        is_dosing = False

    GCODE_text.append(ABBPoint(x=START_X, y=py, z=pz, nx=NX, ny=NY, nz=NZ))

# Stop am Ende
if is_dosing:
    GCODE_text.append(fc.Extruder(on=False))
    GCODE_text.dosingStop()

GCODE_text.append(ABBPoint(x=START_X + SICHERHEITS_X+50, y=mitte_y, z=mitte_z+50, nx=NX, ny=NY, nz=NZ))
GCODE_text.comment("--- ENDE TEXT ---")

visualize_gcode(gcode_list=GCODE_text, frame_filter_value=1)

Ausgabe generiert am: 10.07.2026 um 11:14:05 Uhr
Kein Referenz-GCode-Pfad angegeben. Referenzpfad-Visualisierung übersprungen.


In [ ]:
# --- EXPORT ---
export_gcode(
    gcode_list=GCODE_text,
    file_name="ABB_dosingTestText"
)

GCode erfolgreich exportiert als: ABB_dosingTestText_2026-07-10_11-14.gcode


## Antennenfertigung:

In [ ]:
#@title Helix Test: 08_colabTest_Helix.prg3dp

# Instanziierung der GCode-Liste für die Helix-Geometrie
GCODE_helix = GCodeList()
testHelix_1 = []

# --- GEOMETRISCHE PARAMETER ---
durchmesser = 40       # Außendurchmesser der Helix [mm]
start_winkel = 90      # Startposition der Helix [°]
windung_grad = -270    # Gesamter Rotationswinkel ab Startpunkt [°]
ges_hoehe = 100        # Vertikale Ausdehnung der Helix [mm]
segmente = 100         # Anzahl der Diskretisierungsschritte

# erforderliche RS Ausrichtung T1:[180, 0, -90]; T2:[180, 0, 90]

# --- INITIALISIERUNG UND PROZESSPARAMETER ---
GCODE_helix.comment("--- START Helix PRINT ---")

radius = durchmesser / 2

# Definition des Zentrums basierend auf dem Antennen Referenzpunkt (ANT_ORIGIN)
center = fc.Point(x=ANT_ORIGIN.x, y=ANT_ORIGIN.y, z=ANT_ORIGIN.z+30)

def euler_zyx_to_vectors(yaw: float, pitch: float, roll: float):
    """
    Transformiert ZYX-Euler-Winkel (intrinsisch) in ein orthogonales Vektorsystem.

    Diese Funktion berechnet die Ausrichtung der lokalen X-, Y- und Z-Achsen
    im globalen Raum basierend auf den Rotationswinkeln Yaw (Gier),
    Pitch (Nick) und Roll (Roll).

    Args:
        yaw (float): Rotation um die Z-Achse [rad].
        pitch (float): Rotation um die Y-Achse [rad].
        roll (float): Rotation um die X-Achse [rad].

    Returns:
        tuple: Drei Tupel (vx, vy, vz), welche die Einheitsvektoren der Achsen darstellen.
    """
    cz, sz = math.cos(yaw), math.sin(yaw)
    cy, sy = math.cos(pitch), math.sin(pitch)
    cx, sx = math.cos(roll), math.sin(roll)

    # Berechnung der Richtungsvektoren mittels Rotationsmatrix-Komponenten
    vx = (cz * cy, sz * cy, -sy)
    vy = (cz * sy * sx - sz * cx, sz * sy * sx + cz * cx, cy * sx)
    vz = (cz * sy * cx + sz * sx, sz * sy * cx - cz * sx, cy * cx)

    return vx, vy, vz

# --- GENERIERUNG DER TRAJEKTORIE (LOOP) ---
for i in range(segmente + 1):
    # Berechnung des Fortschrittsfaktors (0.0 bis 1.0)
    fortschritt = i / segmente
    angle = math.radians(start_winkel) + (fortschritt * math.radians(windung_grad))

    # Berechnung der Position in der XY-Ebene mittels Polarkoordinaten
    p = fc.polar_to_point(center, radius, angle)

    # Lineare Interpolation der Höhe über den Fortschritt
    z_pos = center.z + (fortschritt * ges_hoehe)

    # Festlegung der Euler-Winkel für die Werkzeugorientierung:
    # Yaw: Folgt dem aktuellen Windungswinkel (Tangentiale Ausrichtung).
    # Pitch: pi/2 (90 Grad Neigung nach außen, lotrecht zur zentralen Mittelachse).
    # Roll: 0 (Keine Rotation um die Eigenachse).
    vx, vy, vz = euler_zyx_to_vectors(angle, math.pi/2, 0)

    # Erzeugung und Hinzufügen des ABBPoint-Objekts (ohne Frame-Attribut)
    testHelix_1.append(ABBPoint(
        x=p.x, y=p.y, z=z_pos,
        nx=vz[0], ny=vz[1], nz=vz[2]
    ))

GCODE_helix.comment("--- ENDE Helix ---")

GCODE_helix.add_trajectory(testHelix_1)

# Visualisierung aufrufen
visualize_gcode(gcode_list=GCODE_helix, frame_filter_value=1)

Ausgabe generiert am: 10.07.2026 um 11:14:06 Uhr
Kein Referenz-GCode-Pfad angegeben. Referenzpfad-Visualisierung übersprungen.


In [ ]:
# --- EXPORT ---
export_gcode(
    gcode_list=GCODE_helix,
    file_name="ABB_testHelix",
    forceXYZ=True
)

GCode erfolgreich exportiert als: ABB_testHelix_2026-07-10_11-14.gcode


### Definition der Home-Position: `jHomeHelix` in ABB RobotStudio

Die Konstante `jHomeHelix` vom Typ `jointtarget` definiert die festen Gelenkwinkel für diese spezifische Position. Die Zuweisung besteht aus zwei Arrays:

**1. Interne Roboterachsen (in Grad)**
Die ersten sechs Werte definieren die Winkel der Roboterachsen 1 bis 6:
* **Achse 1:** -60°
* **Achse 2:** 25°
* **Achse 3:** 25°
* **Achse 4:** -40°
* **Achse 5:** 75°
* **Achse 6:** -215°

**2. Externe Achsen**
Das zweite Array definiert bis zu sechs externe Zusatzachsen:
* **Werte:** `[9E+09, 9E+09, 9E+09, 9E+09, 9E+09, 9E+09]`
* Der Platzhalterwert `9E+09` gibt an, dass an dieser Position keine externen Achsen verwendet werden.

**RAPID Code - _T_ROB_MAIN_3DP_ :**
```rapid
CONST jointtarget jHomeHelix:=[[-60,25,25,-40,75,-215],[9E+09,9E+09,9E+09,9E+09,9E+09,9E+09]];

PROC main()
        !MoveAbsJ jHomeHelix,v50,fine,T1\WObj:=wobj0;
        %"main3DP"%;
        MoveAbsJ jHome,v200,fine,tool0\WObj:=wobj0;
    ENDPROC
```

**Alternativ die Angabe als Koordinaten:**
```rapid
! Aufbau: [[X, Y, Z], [q1, q2, q3, q4], [Roboter-Konfiguration], [Externe Achsen]]
CONST robtarget pHomeKoor := [[315, 315, 10], [0, 1, 0, 0], [0, 0, 0, 0], [9E+09, 9E+09, 9E+09, 9E+09, 9E+09, 9E+09]];

PROC main()
        MoveJ pHomeKoor,v50,fine,T1\WObj:=DruckbettV4;
        ...
```

----


## Geometriedefintion

  + Außendurchmesser = 26.5mm
  + Innendurchmesser = 24mm
  + Zylinderhöhe = 102mm

  + vertikale Zuleitung = 5mm

  + 4 Arme, 90° versetzt
  + Helix-Höhe = 95mm
  + Turns = 0.75
  + Rechts-Drehend
  + f = 1575,42 MHz (GPS)

  + Kanal b=0.6mm, t=0.4mm
  + Kontaktierung von unten

In [ ]:
#@title Zielgeometrie: QuadHelix GPS Antenne

# 1. GLOBALE DESIGN PARAMETER
AUSSENDURCHMESSER = 26.5       # [mm] Nennmaß des Antennkerns
VERTIKALE_ZULEITUNG = 5.0      # [mm]
ANLAUFSTRECKE = 5.0            # [mm]
HELIX_HOEHE = 95               # [mm]
STARTWINKEL = 156.0            # [°] (6 Uhr = 0°, CCW = positiv)
TURNS = 0.75                   # Gesamtrotationen, LHCP

BASIS_HOEHE = 35               # [mm] Basishöhe von Boden der Aufnahme bis zur Geometrie
KRUEMMUNGSKORREKTUR = 0.000    # [mm] Werkzeugversatz in z-Richtung bei Knickstelle von Zulauf in Helix

KORREKTUR_SINTER_ABSTAND = -0.38 # [mm]
ABLAGEHOEHE_KLEBER = 0.02      # [mm] für V11:0.06

MAX_SEHNENFEHLER = 0.005       # [mm]
CHANNEL_DEPTH = 0.4            # [mm]



# Dynamische Zusammenfassung für den GCode-Header
printData = (
    f"AUSSENDURCHMESSER: {AUSSENDURCHMESSER} mm\n"
    f"VERTIKALE ZULEITUNG: {VERTIKALE_ZULEITUNG} mm\n"
    f"ANLAUFSTRECKE: {ANLAUFSTRECKE} mm\n"
    f"HELIX HOEHE: {HELIX_HOEHE} mm\n"
    f"STARTWINKEL: {STARTWINKEL} DEG\n"
    f"TURNS: {TURNS}\n"
    f"BASIS HOEHE: {BASIS_HOEHE} mm\n"
    f"KRUEMMUNGSKORREKTUR: {KRUEMMUNGSKORREKTUR} mm\n"
    f"SINTER ABSTAND: {KORREKTUR_SINTER_ABSTAND} mm\n"
    f"ABLAGEHOEHE KLEBER: {ABLAGEHOEHE_KLEBER} mm\n"
)



# 2. HILFSFUNKTIONEN
def berechne_kartesisch(cx, cy, radius, user_angle_deg, z_pos):
    """Konvertiert den User-Winkel in globale kartesische Koordinaten basierend auf dem Zentrum."""
    math_rad = math.radians(user_angle_deg - 90)
    x = cx + radius * math.cos(math_rad)
    y = cy + radius * math.sin(math_rad)

    # Normalenvektor zeigt radial nach außen
    nx, ny, nz = math.cos(math_rad), math.sin(math_rad), 0.0
    return x, y, z_pos, nx, ny, nz

def generiere_anfahrtspunkte(eff_durchmesser):
    """
    Generiert drei Punkte, die sich dem Startpunkt in der XY-Ebene radial nähern.
    """
    radius = eff_durchmesser / 2.0
    start_z = ANT_ORIGIN.z - 25 + BASIS_HOEHE - ANLAUFSTRECKE
    x0, y0, z0, nx0, ny0, nz0 = berechne_kartesisch(ANT_ORIGIN.x, ANT_ORIGIN.y, radius, STARTWINKEL, start_z)

    p1 = ABBPoint(x=x0 + nx0*100.0, y=y0 + ny0*100.0, z=z0, nx=nx0, ny=ny0, nz=nz0)
    p2 = ABBPoint(x=x0 + nx0*95.0,  y=y0 + ny0*95.0,  z=z0, nx=nx0, ny=ny0, nz=nz0)
    p3 = ABBPoint(x=x0 + nx0*25.0,  y=y0 + ny0*25.0,  z=z0, nx=nx0, ny=ny0, nz=nz0)
    return [p1, p2, p3]




# 3. ZENTRALE GENERIERUNGSFUNKTION
def generate_helix_path(tool_cmd, speed, eff_durchmesser, anfahrts_punkte, enable_dosing):
    gcode = GCodeList()

    radius = eff_durchmesser / 2.0
    start_z = ANT_ORIGIN.z - 25 + BASIS_HOEHE - ANLAUFSTRECKE

    # 1. Tool-Wechsel und Geschwindigkeit
    #gcode.append(fc.ManualGcode(text=tool_cmd))
    gcode.setSpeed(speed)
    gcode.append(fc.Extruder(on=False))

    # 2. Anfahrtspunkte (p1 -> p2 -> p3)
    for pt in anfahrts_punkte:
        gcode.append(pt)

    # 3. Fahrt auf Startposition
    x0, y0, z0, nx0, ny0, nz0 = berechne_kartesisch(ANT_ORIGIN.x, ANT_ORIGIN.y, radius, STARTWINKEL, start_z)
    gcode.append(ABBPoint(x=x0, y=y0, z=z0-0.1, nx=nx0, ny=ny0, nz=nz0))

    # 4. Pause
    gcode.comment(f"--- START ANT.-KERN ({tool_cmd}) ---")
    gcode.pausePrint()
    gcode.append(fc.Extruder(on=True))
    gcode.append(ABBPoint(x=x0, y=y0, z=z0, nx=nx0, ny=ny0, nz=nz0))

    # 5. Dosierung starten
    if enable_dosing:
        gcode.dosingStart()

    # 6. Senkrechte Fahrt nach oben (+Z)
    z_hoch = start_z + ANLAUFSTRECKE + VERTIKALE_ZULEITUNG + KRUEMMUNGSKORREKTUR
    gcode.append(ABBPoint(x=x0, y=y0, z=z_hoch, nx=nx0, ny=ny0, nz=nz0))

    # 7. Helix abfahren
    gcode.comment(f"--- START Helix ({tool_cmd}) ---")
    winkel_pro_segment_rad = 2 * math.acos(1 - (MAX_SEHNENFEHLER / radius))
    winkel_pro_segment_grad = math.degrees(winkel_pro_segment_rad)
    ges_winkel = TURNS * 360.0
    segmente = math.ceil(ges_winkel / winkel_pro_segment_grad)

    for i in range(1, segmente + 1):
        fortschritt = i / segmente
        curr_angle = STARTWINKEL - (fortschritt * ges_winkel)
        curr_z = z_hoch + (fortschritt * HELIX_HOEHE)

        x, y, z, nx, ny, nz = berechne_kartesisch(ANT_ORIGIN.x, ANT_ORIGIN.y, radius, curr_angle, curr_z)
        gcode.append(ABBPoint(x=x, y=y, z=z, nx=nx, ny=ny, nz=nz))

    last_angle = STARTWINKEL - ges_winkel
    last_z = z_hoch + HELIX_HOEHE

    # 8. Dosierung stoppen
    if enable_dosing:
        gcode.dosingStop()

    # --- RESET-BLOCK ---
    gcode.comment(f"--- ENDE Helix ({tool_cmd}) ---")
    gcode.comment("--- FAHRE ZU START-POS ---")
    gcode.append(fc.Extruder(on=False))

    # 9. 15 mm radial nach außen wegfahren (Z bleibt gleich)
    radius_safe = radius + 15.0
    x_out, y_out, z_out, nx_out, ny_out, nz_out = berechne_kartesisch(ANT_ORIGIN.x, ANT_ORIGIN.y, radius_safe, last_angle, last_z)
    gcode.append(ABBPoint(x=x_out, y=y_out, z=z_out, nx=nx_out, ny=ny_out, nz=nz_out))

    # Geschwindigkeit erhöhen
    return_speed = 7000 - 15 * speed
    gcode.setSpeed(return_speed)

    # 10. Helix mit Sicherheitsabstand rückwärts abfahren
    for i in range(segmente - 1, -1, -1):
        fortschritt = i / segmente
        curr_angle = STARTWINKEL - (fortschritt * ges_winkel)
        curr_z = z_hoch + (fortschritt * HELIX_HOEHE)

        x, y, z, nx, ny, nz = berechne_kartesisch(ANT_ORIGIN.x, ANT_ORIGIN.y, radius_safe, curr_angle, curr_z)
        gcode.append(ABBPoint(x=x, y=y, z=z, nx=nx, ny=ny, nz=nz))

    # 11. Senkrechte Strecke mit Sicherheitsabstand zurück nach unten
    x_safe_start, y_safe_start, z_safe_start, nx_safe_start, ny_safe_start, nz_safe_start = berechne_kartesisch(ANT_ORIGIN.x, ANT_ORIGIN.y, radius_safe, STARTWINKEL, start_z)
    gcode.append(ABBPoint(x=x_safe_start, y=y_safe_start, z=z_safe_start, nx=nx_safe_start, ny=ny_safe_start, nz=nz_safe_start))

    # 12. Anfahrtspunkte rückwärts abfahren (p3 -> p2 -> p1)
    for pt in reversed(anfahrts_punkte):
        gcode.append(pt)

    #gcode.append(fc.ManualGcode(text="T0")) # Standard-Rückwechsel
    return gcode




# 4. WERKZEUGSPEZIFISCHE GENERIERUNG

# Tool 2: Kleber
eff_d_tool2 = AUSSENDURCHMESSER - (2 * CHANNEL_DEPTH) + (2 * ABLAGEHOEHE_KLEBER)
anfahrts_punkte_t2 = generiere_anfahrtspunkte(eff_d_tool2)

gcodeQuadHelixAntenne_V1_Tool2 = generate_helix_path(
    tool_cmd="T1",
    speed=300, # Geschwindigkeit Kleber [mm/min]
    eff_durchmesser=eff_d_tool2,
    anfahrts_punkte=anfahrts_punkte_t2,
    enable_dosing=True
)

# Tool 1: Sintern
eff_d_tool1 = AUSSENDURCHMESSER + (2 * KORREKTUR_SINTER_ABSTAND)
anfahrts_punkte_t1 = generiere_anfahrtspunkte(eff_d_tool1)

gcodeQuadHelixAntenne_V1_Tool1 = generate_helix_path(
    tool_cmd="T0",
    speed=60, # Geschwindigkeit Sintern [mm/min], zusätzlich Bewegungsgeschwindigkeit manuell auf 10% stellen
    eff_durchmesser=eff_d_tool1,
    anfahrts_punkte=anfahrts_punkte_t1,
    enable_dosing=False
)



# 5. VISUALISIERUNG
print("Visualisierung Tool 2 (Kleber):")
visualize_gcode(gcode_list=gcodeQuadHelixAntenne_V1_Tool2, frame_filter_value=1)

print("\n----------------------------------------------------------------------------------------\n\n")

print("Visualisierung Tool 1 (Sintern):")
visualize_gcode(gcode_list=gcodeQuadHelixAntenne_V1_Tool1, frame_filter_value=1)

Visualisierung Tool 2 (Kleber):
Ausgabe generiert am: 10.07.2026 um 11:59:15 Uhr
Kein Referenz-GCode-Pfad angegeben. Referenzpfad-Visualisierung übersprungen.



----------------------------------------------------------------------------------------


Visualisierung Tool 1 (Sintern):
Ausgabe generiert am: 10.07.2026 um 11:59:15 Uhr
Kein Referenz-GCode-Pfad angegeben. Referenzpfad-Visualisierung übersprungen.


In [ ]:
# Export

# Tool 1:
# Ausrichtung [180, 0, -90]
export_gcode(gcode_list=gcodeQuadHelixAntenne_V1_Tool1, file_name="QuadHelixAntenne_V1_Tool1_Sintern", forceXYZ=True, printData=printData)

# Tool 2:
# Ausrichtung [180, 0, 90]
export_gcode(gcode_list=gcodeQuadHelixAntenne_V1_Tool2, file_name="QuadHelixAntenne_V1_Tool2_Kleber", forceXYZ=True, printData=printData)

GCode erfolgreich exportiert als: QuadHelixAntenne_V1_Tool1_Sintern_2026-07-10_12-00.gcode
GCode erfolgreich exportiert als: QuadHelixAntenne_V1_Tool2_Kleber_2026-07-10_12-00.gcode
